# Notebook 01 : DÉFENSEUR (personne A)
**Auteur : ........**  (ce notebook et `src/common.py` sont à vous ; `src/attack.py` et `02_attaquant.ipynb` sont à B)

| Phase | Contenu |
|---|---|
| 1 | données, nettoyage, variables d'URL, découpage sans fuite, audit |
| 2 | baseline, Random Forest, XGBoost, métriques, seuil |
| 5 | durcissement (à faire quand B a livré `src/attack.py`) |

À la fin des phases 1 et 2, ce notebook **sauvegarde l'état** dans `artifacts/` : B en a besoin pour son notebook 02.
Chaque phase finit par un encadré **À écrire dans le rapport**.

**Données** : `PhiUSIIL_Phishing_URL_Dataset.csv` dans `data/` (jamais versionné). Seules les colonnes `URL` et `label` sont lues.

## Phase 0 : configuration

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())          # racine du dépôt, pour `import src`
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.metrics import precision_recall_curve, confusion_matrix, classification_report
from src.common import *                 # configuration, variables d'URL, métriques, modèles, état partagé
print("config OK | graine", SEED, "| mode rapide" if MAX_ROWS else "| jeu complet")

## Phase 1 : données, nettoyage, variables, découpage
### 1.1 Chargement et description

In [ ]:
df = pd.read_csv(DATA_PATH, usecols=[URL_COL, LABEL_COL])
if MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
df["y"] = (df[LABEL_COL] == PHISH_VALUE).astype(int)      # y = 1 : phishing
print("lignes :", len(df))
print(df["y"].value_counts().rename({1: "phishing", 0: "légitime"}))
print(df["y"].value_counts(normalize=True).round(3).rename({1: "phishing", 0: "légitime"}))
print("\nQuelques URL étiquetées phishing (vérifiez que c'est cohérent) :")
print(df[df.y == 1][URL_COL].head(5).to_string(index=False))
print("\nQuelques URL étiquetées légitimes :")
print(df[df.y == 0][URL_COL].head(5).to_string(index=False))
df["y"].value_counts().rename({1: "phishing", 0: "légitime"}).plot(kind="bar", title="Répartition des classes")
plt.tight_layout(); plt.savefig("figures/01_classes.png", dpi=130); plt.show()

### 1.2 Nettoyage (chaque ligne supprimée est comptée et justifiée)

In [ ]:
log = [("lignes initiales", len(df), "")]

n0 = len(df); df = df.dropna(subset=[URL_COL, LABEL_COL])
log.append(("valeurs manquantes (URL ou étiquette)", n0 - len(df), "inutilisables"))

df["url_norm"] = df[URL_COL].astype(str).str.strip()
n0 = len(df); df = df[df["url_norm"].str.len() > 0]
log.append(("URL vides", n0 - len(df), "inutilisables"))

df["url_key"] = df["url_norm"].str.lower().str.rstrip("/")     # détecte les quasi-doublons

n_lab = df.groupby("url_key")["y"].nunique()                    # même URL, étiquettes contradictoires
conflict = set(n_lab[n_lab > 1].index)
n0 = len(df); df = df[~df["url_key"].isin(conflict)]
log.append(("URL avec étiquettes contradictoires", n0 - len(df), "ambiguës"))

n0 = len(df); df = df.drop_duplicates(subset="url_key").reset_index(drop=True)   # AVANT le découpage
log.append(("doublons (casse et '/' final ignorés)", n0 - len(df), "fuite train/test sinon"))

log.append(("lignes finales", len(df), ""))
log_df = pd.DataFrame(log, columns=["étape", "lignes", "raison"])
log_df.to_csv("results/nettoyage.csv", index=False)
log_df

### 1.3 Variables calculées à partir de la chaîne d'URL
Les variables sont définies dans `src/common.py` (fonction `url_features`). Elles ne dépendent que de l'URL : aucune variable de contenu de page
ni de score calculé par les auteurs du jeu (`URLSimilarityIndex`, `TLDLegitimateProb`… ne sont **pas** utilisées).

In [ ]:
df["domain"] = df["url_norm"].map(reg_domain)
t0 = time.time()
X_all = featurize(df["url_norm"])
print(f"{X_all.shape[1]} variables, {len(X_all)} URL, calcul en {time.time()-t0:.0f}s")
print("Variables :", FEATURES)
X_all.head()

### 1.4 Découpage sans fuite (par domaine) et sauvegarde de l'état
Tous les URL d'un même domaine vont dans le même jeu. 60 % entraînement, 20 % validation, 20 % test.

In [ ]:
y_all, groups_all = df["y"].values, df["domain"].values
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
trval_idx, test_idx = next(gss1.split(X_all, y_all, groups_all))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)   # 25 % de 80 % = 20 %
tr_rel, va_rel = next(gss2.split(X_all.iloc[trval_idx], y_all[trval_idx], groups_all[trval_idx]))
train_idx, val_idx = trval_idx[tr_rel], trval_idx[va_rel]

S = make_state(df[["url_norm", "y", "domain"]], train_idx, val_idx, test_idx, X_all=X_all)
X_train, y_train, g_train = S.X_train, S.y_train, S.g_train
X_val,   y_val,   g_val   = S.X_val,   S.y_val,   S.g_val
X_test,  y_test,  g_test  = S.X_test,  S.y_test,  S.g_test

# contrôles anti-fuite : ces lignes doivent passer
assert not (set(g_train) & set(g_val)),  "domaines communs train/val"
assert not (set(g_train) & set(g_test)), "domaines communs train/test"
assert not (set(g_val) & set(g_test)),   "domaines communs val/test"
assert not (set(S.urls_train) & set(S.urls_test)), "URL communes train/test"

save_state(S)       # écrit artifacts/clean.csv.gz et les indices : B en a besoin
pd.DataFrame({"jeu": ["train", "val", "test"],
              "lignes": [len(train_idx), len(val_idx), len(test_idx)],
              "% phishing": [y_train.mean().round(3), y_val.mean().round(3), y_test.mean().round(3)],
              "domaines": [len(set(g_train)), len(set(g_val)), len(set(g_test))]})

### 1.5 Audit des variables (artefacts du jeu)
Une variable qui sépare presque parfaitement les classes **seule** est suspecte : elle reflète peut-être la façon dont le jeu a été collecté
(par exemple `https://www.` dans presque toutes les URL légitimes). Calculé sur le **train** uniquement.

In [ ]:
X_aud = featurize(S.urls_train, cols=ALL_FEATURES)          # audit sur TOUTES les variables, même retirées
aud = []
for f in ALL_FEATURES:
    a = roc_auc_score(y_train, X_aud[f])
    aud.append((f, round(a, 3), round(max(a, 1 - a), 3)))
aud = pd.DataFrame(aud, columns=["variable", "AUC (sens brut)", "pouvoir séparateur"])
aud["utilisée par le modèle"] = aud["variable"].isin(FEATURES)
aud = aud.sort_values("pouvoir séparateur", ascending=False)
aud.to_csv("results/audit_variables.csv", index=False)
print("Variables les plus séparatrices seules (suspectes si > 0,95) :")
aud.head(10)

**Deux profils de détecteur** (variable d'environnement `PROFILE`, voir README) :
- `v1` : les 25 variables d'origine. Détecteur « facile » : il exploite des raccourcis du jeu (`is_https`, `has_www`, longueur du chemin).
- `v2` (défaut) : on **retire `is_https` et `has_www`** et on **ajoute 9 variables** (mots sensibles, entropie et aspect du nom de domaine, séries de chiffres...).
  Le détecteur doit alors reconnaître le phishing par ce qu'il contient, pas par un artefact de collecte.

La colonne « utilisée par le modèle » du tableau ci-dessus montre ce qui est retiré. **Prévenez B** si vous changez de profil : ses résultats en dépendent.

> **À écrire dans le rapport (phase 1)** : tableau de nettoyage, graphique des classes, liste des variables et justification, découpage par domaine et pourquoi,
> résultat de l'audit et décision sur les variables suspectes.

## Phase 2 : modèles et évaluation
### 2.1 Les trois modèles
Baseline = régression logistique. Random Forest et XGBoost sont réglés par validation croisée **par groupes** sur un sous-échantillon du train,
puis réentraînés sur tout le train.

In [ ]:
rng = np.random.RandomState(SEED)
tune_idx = rng.choice(len(X_train), size=min(TUNE_SAMPLE, len(X_train)), replace=False)
cv = GroupKFold(n_splits=3)

BEST, MODELS, FIT_S = {"lr": {}}, {}, {}
for name in ["lr", "rf", "xgb"]:
    if name in GRIDS:
        search = RandomizedSearchCV(BUILDERS[name](), GRIDS[name], n_iter=6, scoring="average_precision",
                                    cv=cv, random_state=SEED, n_jobs=1, error_score="raise")
        search.fit(X_train.iloc[tune_idx], y_train[tune_idx], groups=g_train[tune_idx])
        BEST[name] = search.best_params_
        print(name, "meilleurs paramètres :", BEST[name], "| PR-AUC CV :", round(search.best_score_, 4))
    t0 = time.time()
    MODELS[name] = BUILDERS[name](**BEST[name]).fit(X_train, y_train)
    FIT_S[name] = time.time() - t0

### 2.2 Comparaison sur la validation, seuil, sauvegarde
Seuil d'alerte : choisi **sur la validation**, au plus 1 % de légitimes alertés. Alerte = score ≥ seuil.

In [ ]:
THR, rows = {}, []
for name, m in MODELS.items():
    s_val = m.predict_proba(X_val)[:, 1]
    THR[name] = thr_at_fpr(y_val, s_val)
    r = metrics_at(y_val, s_val, THR[name])
    t0 = time.time(); m.predict_proba(X_val.iloc[:5000]); r_t = (time.time() - t0) / 5000 * 1e6
    rows.append(dict(modèle=name, **{k: round(v, 4) for k, v in r.items() if k in ("pr_auc", "roc_auc", "precision", "recall", "f1", "fpr")},
                     seuil=round(THR[name], 4), fit_s=round(FIT_S[name], 1), infer_ms_par_1000=round(r_t, 1)))
cmp_val = pd.DataFrame(rows).set_index("modèle")
cmp_val.to_csv("results/comparaison_modeles_val.csv")
save_detectors(MODELS, THR, BEST)          # écrit artifacts/detectors.joblib : B en a besoin
cmp_val

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for name, m in MODELS.items():
    p, r_, _ = precision_recall_curve(y_val, m.predict_proba(X_val)[:, 1])
    ax.plot(r_, p, label=f"{name} (PR-AUC {cmp_val.loc[name, 'pr_auc']:.3f})")
ax.axhline(y_val.mean(), ls=":", c="grey", label="hasard (proportion de phishing)")
ax.set_xlabel("Rappel"); ax.set_ylabel("Précision"); ax.set_title("Courbes précision-rappel (validation)"); ax.legend()
plt.tight_layout(); plt.savefig("figures/02_pr_curves_val.png", dpi=130); plt.show()

In [ ]:
m = MODELS[TARGET_NAME]
pred = (m.predict_proba(X_val)[:, 1] >= THR[TARGET_NAME]).astype(int)
print("Détecteur ciblé par l'attaque :", TARGET_NAME, "| seuil :", round(THR[TARGET_NAME], 4))
print("Matrice de confusion [lignes = réel (légitime, phishing) ; colonnes = prédit] :\n", confusion_matrix(y_val, pred))
print(classification_report(y_val, pred, target_names=["légitime", "phishing"], digits=4))

> **À écrire dans le rapport (phase 2)** : tableau de comparaison, courbes PR, matrice de confusion.
> Discutez le **coût des erreurs** : un faux négatif laisse passer un phishing (utilisateur piégé) ; un faux positif bloque un site légitime (fatigue d'alerte, perte de confiance).
> Expliquez pourquoi un seuil à 1 % de faux positifs et ce qui changerait avec 0,1 % ou 5 %. Si les scores sont proches de 1, revenez à l'audit 1.5 avant de conclure.

**Jalon du 12 novembre** : rapport d'une page = phase 1 + cette phase 2 (baseline comprise) + le protocole d'évaluation.
**Prévenez B** : l'état est prêt dans `artifacts/` (il peut lancer le notebook 02).

## Phase 5 : durcissement
**À faire seulement quand B a livré `src/attack.py` (fonctions `greedy_attack`, `set_bounds`).**
On attaque des phishing **d'entraînement** (jamais la validation ni le test), on ajoute les URL modifiées au train avec l'étiquette « phishing »,
on réentraîne avec les mêmes hyperparamètres, en 2 tours. Le seuil est recalculé sur la validation propre (même taux de faux positifs de 1 %).

In [ ]:
from src.attack import greedy_attack, set_bounds
set_bounds(S.hi)
target = Detector(TARGET_NAME, MODELS[TARGET_NAME], THR[TARGET_NAME])

def adversarial_round(det, round_id, n=N_TRAIN_ADV, budget=B_REF):
    sel = sample_phish(S.urls_train, S.y_train, n, SEED + 100 + round_id)
    detected = det.score(sel) >= det.thr
    adv = []
    for u in sel[detected]:
        r = greedy_attack(det, u, budget)
        if r["adv"] != u: adv.append(r["adv"])
    print(f"tour {round_id} : {int(detected.sum())} phishing détectés attaqués, {len(adv)} exemples adverses ajoutés")
    return adv

t0 = time.time()
adv_urls, det_h = [], target
for rnd in (1, 2):
    adv_urls += adversarial_round(det_h, rnd)
    X_aug = pd.concat([X_train, featurize(adv_urls)], ignore_index=True)
    y_aug = np.concatenate([y_train, np.ones(len(adv_urls), dtype=int)])
    m_h = BUILDERS[TARGET_NAME](**BEST[TARGET_NAME]).fit(X_aug, y_aug)
    thr_h = thr_at_fpr(y_val, m_h.predict_proba(X_val)[:, 1])
    det_h = Detector(TARGET_NAME + "_durci", m_h, thr_h)
print(f"durcissement : {time.time()-t0:.0f}s ; seuil durci = {thr_h:.4f}")
hardened = det_h
save_hardened(hardened)        # écrit artifacts/hardened.joblib : nécessaire au notebook 03

### 5.2 Coût du durcissement sur les performances normales (validation)

In [ ]:
normal_cmp = pd.DataFrame([summary_row(d.name, y_val, d.score_F(X_val), d.thr) for d in (target, hardened)]).set_index("modèle")
normal_cmp.to_csv("results/performance_normale_avant_apres.csv")
normal_cmp

### 5.3 Option B : règles de validation d'entrée
Signaler les URL dont une variable sort de l'intervalle observé en entraînement ; on mesure combien de vraies URL légitimes cette règle bloque à tort.

In [ ]:
def out_of_range(F): return ~(F[FEATURES].values <= S.hi[FEATURES].values).all(axis=1)
print("légitimes bloquées à tort par la règle :", round(float(out_of_range(X_val[y_val == 0]).mean()), 4))
print("phishing attrapés par la règle seule   :", round(float(out_of_range(X_val[y_val == 1]).mean()), 4))
print("Note : les attaques de B restent dans les bornes par construction ; cette règle ne les voit donc pas. À discuter dans le rapport.")

> **À écrire dans le rapport (phase 5)** : méthode (qui attaque quoi, combien de tours), tableau `results/performance_normale_avant_apres.csv`,
> nombre d'exemples adverses ajoutés, résultat de l'option B. Dites clairement si le durcissement a coûté de la performance normale.
> Les courbes avant/après sont dans le notebook 03.